<a href="https://colab.research.google.com/github/rishithareddym/AI-Lab/blob/main/1BF24CS181_WEEK8_FORWARD_CHAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import itertools

def is_variable(x):
    return isinstance(x, str) and x.startswith('?')

def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    elif is_variable(x) and x in theta:
        return unify(var, theta[x], theta)
    else:
        new_theta = theta.copy()
        new_theta[var] = x
        return new_theta

def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if theta is False:
        return False
    elif x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, tuple) and isinstance(y, tuple):
        if len(x) != len(y):
            return False
        for a, b in zip(x, y):
            theta = unify(a, b, theta)
            if theta is False:
                return False
        return theta
    else:
        return False

def subst(theta, expr):
    if isinstance(expr, str):
        return theta.get(expr, expr)
    elif isinstance(expr, tuple):
        return tuple(subst(theta, arg) for arg in expr)
    return expr

var_counter = 0

def standardize_variables(rule):
    global var_counter
    var_counter += 1

    premises, conclusion = rule

    def extract_vars(expr):
        if is_variable(expr):
            return {expr}
        elif isinstance(expr, tuple):
            vars_set = set()
            for arg in expr:
                vars_set.update(extract_vars(arg))
            return vars_set
        return set()

    all_vars = set()
    for p in premises:
        all_vars.update(extract_vars(p))
    all_vars.update(extract_vars(conclusion))

    var_map = {v: f"?v_{var_counter}_{v[1:]}" for v in all_vars}

    def apply_map(expr):
        if is_variable(expr):
            return var_map[expr]
        elif isinstance(expr, tuple):
            return tuple(apply_map(arg) for arg in expr)
        return expr

    standardized_premises = tuple(apply_map(p) for p in premises)
    standardized_conclusion = apply_map(conclusion)
    return (standardized_premises, standardized_conclusion)

def fol_fc_ask(KB_facts, KB_rules, alpha):
    KB = set(KB_facts)

    while True:
        new = set()

        for rule in KB_rules:
            premises, q = standardize_variables(rule)
            n = len(premises)

            for fact_combination in itertools.product(KB, repeat=n):
                theta = {}
                unified = True

                for p_i, fact_i in zip(premises, fact_combination):
                    theta = unify(subst(theta, p_i), fact_i, theta)
                    if theta is False:
                        unified = False
                        break

                if unified:
                    q_prime = subst(theta, q)

                    already_exists = any(
                        unify(q_prime, sentence, {}) is not False
                        for sentence in KB.union(new)
                    )

                    if not already_exists:
                        new.add(q_prime)

                        phi = unify(q_prime, alpha, {})

                        if phi is not False:
                            return phi

        if not new:
            break

        KB.update(new)

    return False


facts = {
    ('American', 'Robert'),
    ('Enemy', 'Nono', 'America'),
    ('Missile', 'M1'),
    ('Owns', 'Nono', 'M1'),
    ('Sells', 'Robert', 'M1', 'Nono')
}

rules = [
    ((('Missile', '?x'),), ('Weapon', '?x')),
    ((('Enemy', '?x', 'America'),), ('Hostile', '?x')),
    (
        (('American', '?x'), ('Weapon', '?y'), ('Sells', '?x', '?y', '?z'), ('Hostile', '?z')),
        ('Criminal', '?x')
    )
]

alpha = ('Criminal', '?person')

result = fol_fc_ask(facts, rules, alpha)
print("Resulting substitution:", result)

Resulting substitution: {'?person': 'Robert'}
